# CGCNN Training Notebook

This notebook allows you to interactively train the Crystal Graph Convolutional Neural Network (CGCNN) on your material data.

In [10]:
import os
import sys
import random
import logging
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

# Add parent directory to path to load 'training' module
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, "../.."))
if project_root not in sys.path:
    sys.path.append(project_root)

# Import custom modules (ensure these exist in mattergen-x/training)
from training.datasets.loader import MaterialDataLoader
from training.datasets.graph_builder import CrystalGraphBuilder
from training.datasets.graph_dataset import CrystalGraphDataset, collate_batch
from training.models.cgcnn import CGCNN

# Configure Logger
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger()

ModuleNotFoundError: No module named 'numpy'

In [8]:
# --- Configuration ---
class Config:
    # Data paths (Adjust relative to notebook location)
    # Pointing to the NEW Clean Dataset (MP-20)
    data_path = "../../data/datasets/mp20_clean.json"
    save_dir = "../../models"
    model_name = "cgcnn_best.pt"
    
    # Training Hyperparameters
    epochs = 100
    batch_size = 64 # Increased batch size for larger dataset stability
    learning_rate = 1e-3 # Lower LR for stability
    patience = 20
    seed = 42

args = Config()
os.makedirs(args.save_dir, exist_ok=True)
print(f"Model will be saved to: {os.path.abspath(args.save_dir)}")

Model will be saved to: d:\coding_files\Projects\matterGen\mattergen-x\models


In [9]:
def set_seed(seed: int = 42):
    """Ensure reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(args.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

NameError: name 'np' is not defined

## 1. Data Loading & Graph Construction

In [ ]:
print(f"Loading data from {args.data_path}...")
loader = MaterialDataLoader(args.data_path)
raw_data = loader.load()

if not raw_data:
    # Fallback/Debug if file not found
    print("Warning: No data loaded via loader. Checking path...")
    if not os.path.exists(args.data_path):
        print(f"File not found at {args.data_path}")
    else:
        print(f"File exists but loader returned empty. Content: {raw_data}")
else:
    print(f"Loaded {len(raw_data)} samples.")

# Initialize Graph Builder
# radius=8.0, dStep=0.2 -> 41 edge features
builder = CrystalGraphBuilder(radius=8.0, dStep=0.2)

# Build Dataset (this might take a few seconds)
print("Building Graph Dataset... (This may take a minute for 45k samples)")
full_dataset = CrystalGraphDataset(raw_data, builder, cache_graphs=True)

In [ ]:
# Split Train/Val/Test
indices = list(range(len(full_dataset)))
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42)
train_idx, val_idx = train_test_split(train_idx, test_size=0.1, random_state=42)

train_set = Subset(full_dataset, train_idx)
val_set = Subset(full_dataset, val_idx)
test_set = Subset(full_dataset, test_idx)

print(f"Train size: {len(train_set)}")
print(f"Val size:   {len(val_set)}")
print(f"Test size:  {len(test_set)}")

In [ ]:
# Normalize Targets
train_targets = []
for i in train_idx:
    train_targets.append(full_dataset[i]['target'].numpy())
train_targets = np.array(train_targets)

scaler = StandardScaler()
scaler.fit(train_targets)

target_mean = torch.tensor(scaler.mean_, device=device, dtype=torch.float32)
target_scale = torch.tensor(scaler.scale_, device=device, dtype=torch.float32)

print(f"Target Mean: {target_mean}")
print(f"Target Std:  {target_scale}")

In [ ]:
train_loader = DataLoader(train_set, batch_size=args.batch_size, shuffle=True, collate_fn=collate_batch)
val_loader = DataLoader(val_set, batch_size=args.batch_size, shuffle=False, collate_fn=collate_batch)
test_loader = DataLoader(test_set, batch_size=args.batch_size, shuffle=False, collate_fn=collate_batch)

## 2. Model Initialization

In [ ]:
model = CGCNN(
    node_input_dim=4,     # Z, Group, Period, X
    edge_input_dim=41,    # From gaussian expansion
    node_hidden_dim=64, 
    n_conv_layers=3, 
    n_targets=3           # Formation Energy, Band Gap, Density
).to(device)

optimizer = optim.Adam(model.parameters(), lr=args.learning_rate)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)
criterion = nn.L1Loss() # MAE Loss

print(model)

## 3. Training Loop

In [ ]:
history = {'train_loss': [], 'val_loss': []}
best_val_mae = float('inf')
early_stop_counter = 0
save_path = os.path.join(args.save_dir, args.model_name)

print("Starting training...")

for epoch in range(args.epochs):
    # --- Training ---
    model.train()
    train_loss = 0.0
    
    for batch in train_loader:
        atom_fea = batch['atom_fea'].to(device)
        nbr_fea = batch['nbr_fea'].to(device)
        nbr_idx = batch['nbr_idx'].to(device)
        batch_map = batch['batch'].to(device)
        targets = batch['target'].to(device)
        
        targets_norm = (targets - target_mean) / target_scale
        
        optimizer.zero_grad()
        preds_norm = model(atom_fea, nbr_fea, nbr_idx, batch_map)
        
        loss = criterion(preds_norm, targets_norm)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * targets.size(0)
        
    train_loss /= len(train_set)

    # --- Validation ---
    model.eval()
    val_loss = 0.0
    val_mae_sum = torch.zeros(3, device=device)
    total_val = 0
    
    with torch.no_grad():
        for batch in val_loader:
            atom_fea = batch['atom_fea'].to(device)
            nbr_fea = batch['nbr_fea'].to(device)
            nbr_idx = batch['nbr_idx'].to(device)
            batch_map = batch['batch'].to(device)
            targets = batch['target'].to(device)
            
            preds_norm = model(atom_fea, nbr_fea, nbr_idx, batch_map)
            targets_norm = (targets - target_mean) / target_scale
            
            loss = criterion(preds_norm, targets_norm)
            val_loss += loss.item() * targets.size(0)
            
            # Un-normalized MAE
            preds = preds_norm * target_scale + target_mean
            val_mae_sum += torch.sum(torch.abs(preds - targets), dim=0)
            total_val += targets.size(0)

    val_loss /= len(val_set)
    val_maes = val_mae_sum / total_val
    avg_val_mae = torch.mean(val_maes).item()
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    
    scheduler.step(val_loss)

    # Logging
    if epoch % 5 == 0:
        print(f"Epoch {epoch+1}/{args.epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
        print(f"   >> Val MAE per target: E_f={val_maes[0]:.3f}, Bg={val_maes[1]:.3f}, Dens={val_maes[2]:.3f}")
    
    # Checkpointing
    if avg_val_mae < best_val_mae:
        best_val_mae = avg_val_mae
        early_stop_counter = 0
        torch.save({
            'model_state_dict': model.state_dict(),
            'scaler_mean': target_mean,
            'scaler_scale': target_scale
        }, save_path)
    else:
        early_stop_counter += 1
        if early_stop_counter >= args.patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

print("Training Complete.")
print(f"Best Val MAE: {best_val_mae:.4f}")

## 4. Evaluation

In [ ]:
# Plot Loss
plt.figure(figsize=(10, 5))
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss (MAE)')
plt.title('Training Trajectory')
plt.legend()
plt.show()

In [ ]:
# Load Best Model for Testing
checkpoint = torch.load(save_path)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

test_mae_sum = torch.zeros(3, device=device)
total_test = 0

with torch.no_grad():
    for batch in test_loader:
        atom_fea = batch['atom_fea'].to(device)
        nbr_fea = batch['nbr_fea'].to(device)
        nbr_idx = batch['nbr_idx'].to(device)
        batch_map = batch['batch'].to(device)
        targets = batch['target'].to(device)
        
        preds_norm = model(atom_fea, nbr_fea, nbr_idx, batch_map)
        preds = preds_norm * target_scale + target_mean
        
        test_mae_sum += torch.sum(torch.abs(preds - targets), dim=0)
        total_test += targets.size(0)

test_maes = test_mae_sum / total_test

print("TEST RESULTS (MAE):")
print(f"Formation Energy: {test_maes[0]:.4f} eV/atom")
print(f"Band Gap:         {test_maes[1]:.4f} eV")
print(f"Density:          {test_maes[2]:.4f} g/cm^3")